# LeetCode 1822 — Sign of the Product of an Array bằng Python

Link đề: https://leetcode.com/problems/sign-of-the-product-of-an-array/

Độ khó: **Easy** · Chủ đề: Array, Math

```python
class Solution:
    def arraySign(self, nums: List[int]) -> int:

```

**Mục tiêu học:**

1. Viết hàm dấu `signFunc(x)` với `if / elif / else`.
2. Hiểu vì sao tính tích trực tiếp có thể **tràn số** (ở C++/Java) hoặc chậm với số rất lớn.
3. Rút ra quy tắc toán: dấu của tích chỉ phụ thuộc **có số 0 không** và **số lượng số âm chẵn hay lẻ**.
4. Viết lời giải **lật dấu** `sign = -sign` với **dừng sớm** khi gặp 0.
5. Biết `math.prod`, `functools.reduce`, `sum(...)` với generator để đếm.

## 1. Bài toán yêu cầu gì?

Định nghĩa `signFunc(x)`:

- `1` nếu `x > 0`
- `-1` nếu `x < 0`
- `0` nếu `x == 0`

Cho mảng số nguyên `nums`, gọi `product` là **tích** mọi phần tử. Trả về `signFunc(product)`.

| `nums` | Tích | Kết quả |
|---|---|:---:|
| `[-1,-2,-3,-4,3,2,1]` | 144 | `1` |
| `[1,5,0,2,-3]` | 0 | `0` |
| `[-1,1,-1,1,-1]` | -1 | `-1` |
| `[7]` | 7 | `1` |
| `[-100] * 1000` | 10^2000 (dương) | `1` |

**Ràng buộc:** `1 <= len(nums) <= 1000`, `-100 <= nums[i] <= 100`.

Chú ý dòng cuối bảng: tích có thể lên tới `100^1000` = số có **2001 chữ số**!

In [1]:
from typing import List

examples = [
    ([-1, -2, -3, -4, 3, 2, 1], 1),
    ([1, 5, 0, 2, -3], 0),
    ([-1, 1, -1, 1, -1], -1),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[-1, -2, -3, -4, 3, 2, 1] -> kỳ vọng 1
[1, 5, 0, 2, -3] -> kỳ vọng 0
[-1, 1, -1, 1, -1] -> kỳ vọng -1


## 2. Nền tảng Python cần biết

### 2.1. `if / elif / else` — viết `signFunc`

`elif` = "else if": chỉ được xét khi các điều kiện trước **sai**. Đúng một nhánh được chạy.

In [2]:
def sign_func(x: int) -> int:
    if x > 0:
        return 1
    elif x < 0:
        return -1
    else:
        return 0

for x in [144, 0, -1, -5000]:
    print(x, "->", sign_func(x))

144 -> 1
0 -> 0
-1 -> -1
-5000 -> -1


### 2.2. Tích một list: vòng lặp, `math.prod`, `functools.reduce`

- Vòng lặp: `p = 1; for x in nums: p *= x` (khởi tạo bằng **1**, phần tử trung hòa của phép nhân — không phải 0!).
- `math.prod(nums)` (Python 3.8+) làm sẵn việc đó.
- `functools.reduce(f, nums, 1)` áp dụng `f` gộp dần từ trái sang.

In [3]:
import math
from functools import reduce

nums = [-1, -2, -3, -4, 3, 2, 1]
p = 1
for x in nums:
    p *= x
print("vòng lặp:", p)
print("math.prod:", math.prod(nums))
print("reduce:", reduce(lambda a, b: a * b, nums, 1))

vòng lặp: 144
math.prod: 144
reduce: 144


### 2.3. Số nguyên lớn và vấn đề tràn số

Python tự mở rộng số nguyên nên không tràn, nhưng ở C++/Java `int` chỉ 32 bit (~2.1 × 10^9), `long` 64 bit (~9.2 × 10^18). Tích 1000 số cỡ 100 vượt xa giới hạn đó.

In [4]:
big = math.prod([-100] * 1000)
print("số chữ số của |tích|:", len(str(abs(big))))
print("giới hạn int 32-bit:", 2**31 - 1)
print("giới hạn long 64-bit:", 2**63 - 1)

số chữ số của |tích|: 2001
giới hạn int 32-bit: 2147483647
giới hạn long 64-bit: 9223372036854775807


### 2.4. Đếm bằng `sum` + generator, và `%` để xét chẵn lẻ

`sum(1 for x in nums if x < 0)` đếm số phần tử âm. `count % 2` là `0` nếu chẵn, `1` nếu lẻ. `0 in nums` kiểm tra có phần tử 0 không.

In [5]:
nums = [-1, -2, -3, -4, 3, 2, 1]
neg = sum(1 for x in nums if x < 0)
print("số âm:", neg, "| chẵn?", neg % 2 == 0)
print("có số 0?", 0 in nums, "|", 0 in [1, 5, 0, 2, -3])

số âm: 4 | chẵn? True
có số 0? False | True


## 3. Cách trực giác — nhân hết rồi lấy dấu

Làm đúng như đề nói: tính `product`, gọi `sign_func(product)`.

In [6]:
def array_sign_product(nums: List[int]) -> int:
    product = 1
    for x in nums:
        product *= x
    return sign_func(product)

for nums, expected in examples:
    print(array_sign_product(nums), "| kỳ vọng", expected)

1 | kỳ vọng 1
0 | kỳ vọng 0
-1 | kỳ vọng -1


### 3.1. Vấn đề của cách nhân trực tiếp

- Trong Python: **đúng**, nhưng nhân số ngày càng lớn (hàng nghìn chữ số) tốn thời gian hơn `O(1)` mỗi phép.
- Ở ngôn ngữ khác: **tràn số** → kết quả sai hoàn toàn (thậm chí đổi dấu).
- Ta tính một con số khổng lồ, trong khi chỉ cần **một trong ba giá trị** `-1, 0, 1`.

Gợi ý của đề: *"Nếu có số 0 thì đáp án là 0. Để tránh tràn số, hãy biến mọi số âm thành -1 và mọi số dương thành 1 rồi nhân."*

In [7]:
# Mô phỏng tràn số 32-bit có dấu (giống int của C/Java) để thấy kết quả sai
def to_int32(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= (1 << 31) else x

p32 = 1
for x in [100] * 6:
    p32 = to_int32(p32 * x)       # mỗi bước nhân đều bị cắt về 32 bit như C/Java
print("tích thật 100^6 =", 100**6, "-> dấu:", sign_func(100**6))
print("tích 32-bit tràn =", p32, "-> dấu:", sign_func(p32), "(SAI, đáng lẽ là 1)")
assert sign_func(p32) != sign_func(100**6)

tích thật 100^6 = 1000000000000 -> dấu: 1
tích 32-bit tràn = -727379968 -> dấu: -1 (SAI, đáng lẽ là 1)


(Hàm `to_int32` chỉ để minh họa: `& 0xFFFFFFFF` giữ 32 bit thấp, rồi diễn giải lại thành số có dấu. Không cần nhớ chi tiết.)

## 4. Nút thắt và insight

Quy tắc dấu của phép nhân mà ta học ở cấp 2:

- Có **ít nhất một số 0** → tích bằng 0.
- Không có 0: mỗi số âm "lật" dấu một lần. Số lượng số âm **chẵn** → dương; **lẻ** → âm.

> **Insight:** Không cần độ lớn, chỉ cần theo dõi **dấu**. Giữ `sign ∈ {1, -1}`, gặp số âm thì `sign = -sign`, gặp 0 thì trả 0 ngay.

In [8]:
for nums in [[-1, -2, -3, -4, 3, 2, 1], [-1, 1, -1, 1, -1], [1, 5, 0, 2, -3]]:
    neg = sum(1 for x in nums if x < 0)
    has_zero = 0 in nums
    print(nums, "| số âm =", neg, "| có 0:", has_zero, "| tích =", math.prod(nums))

[-1, -2, -3, -4, 3, 2, 1] | số âm = 4 | có 0: False | tích = 144
[-1, 1, -1, 1, -1] | số âm = 3 | có 0: False | tích = -1
[1, 5, 0, 2, -3] | số âm = 1 | có 0: True | tích = 0


## 5. Lời giải chính — lật dấu, dừng sớm khi gặp 0

**Bước 1.** `sign = 1`.

**Bước 2.** Với mỗi `x`:
- `x == 0` → `return 0` (không cần xét tiếp).
- `x < 0` → `sign = -sign`.
- `x > 0` → không làm gì.

**Bước 3.** `return sign`.

In [9]:
# Đoạn nhỏ: lật dấu nhiều lần
sign = 1
for step in range(4):
    sign = -sign
    print("sau lần lật", step + 1, ":", sign)

sau lần lật 1 : -1
sau lần lật 2 : 1
sau lần lật 3 : -1
sau lần lật 4 : 1


In [10]:
def array_sign(nums: List[int]) -> int:
    sign = 1
    for x in nums:
        if x == 0:
            return 0
        if x < 0:
            sign = -sign
    return sign

for nums, expected in examples:
    print(array_sign(nums), "| kỳ vọng", expected)

1 | kỳ vọng 1
0 | kỳ vọng 0
-1 | kỳ vọng -1


## 6. TRACE từng bước

Trace `nums = [-1, 1, -1, 1, -1]`:

| i | x | Nhánh | sign sau |
|---|---|---|---|
| – | – | khởi tạo | 1 |
| 0 | -1 | âm → lật | -1 |
| 1 | 1 | dương | -1 |
| 2 | -1 | âm → lật | 1 |
| 3 | 1 | dương | 1 |
| 4 | -1 | âm → lật | **-1** |

Trace `nums = [1, 5, 0, 2, -3]`: tới `i = 2` gặp `0` → trả `0` ngay, **không** xét `2, -3`.

In [11]:
def trace_array_sign(nums: List[int]) -> int:
    print("nums =", nums)
    sign = 1
    print("khởi tạo sign = 1")
    for i, x in enumerate(nums):
        if x == 0:
            print(f"i={i}: x=0 -> tích chắc chắn bằng 0, return 0 (bỏ qua {len(nums) - i - 1} phần tử còn lại)")
            return 0
        if x < 0:
            sign = -sign
            print(f"i={i}: x={x} âm -> lật dấu, sign = {sign}")
        else:
            print(f"i={i}: x={x} dương -> giữ sign = {sign}")
    print("-> return", sign)
    return sign

trace_array_sign([-1, 1, -1, 1, -1])
print()
trace_array_sign([1, 5, 0, 2, -3])

nums = [-1, 1, -1, 1, -1]
khởi tạo sign = 1
i=0: x=-1 âm -> lật dấu, sign = -1
i=1: x=1 dương -> giữ sign = -1
i=2: x=-1 âm -> lật dấu, sign = 1
i=3: x=1 dương -> giữ sign = 1
i=4: x=-1 âm -> lật dấu, sign = -1
-> return -1

nums = [1, 5, 0, 2, -3]
khởi tạo sign = 1
i=0: x=1 dương -> giữ sign = 1
i=1: x=5 dương -> giữ sign = 1
i=2: x=0 -> tích chắc chắn bằng 0, return 0 (bỏ qua 2 phần tử còn lại)


0

### 6.1. Trace ví dụ 1 (bốn số âm → dương)

In [12]:
trace_array_sign([-1, -2, -3, -4, 3, 2, 1])

nums = [-1, -2, -3, -4, 3, 2, 1]
khởi tạo sign = 1
i=0: x=-1 âm -> lật dấu, sign = -1
i=1: x=-2 âm -> lật dấu, sign = 1
i=2: x=-3 âm -> lật dấu, sign = -1
i=3: x=-4 âm -> lật dấu, sign = 1
i=4: x=3 dương -> giữ sign = 1
i=5: x=2 dương -> giữ sign = 1
i=6: x=1 dương -> giữ sign = 1
-> return 1


1

## 7. Các lời giải khác

### Cách 3 — Đếm số âm, kiểm tra 0

Hai lượt duyệt (một cho `0 in nums`, một để đếm), nhưng code rất "toán học".

In [13]:
def array_sign_count(nums: List[int]) -> int:
    if 0 in nums:
        return 0
    negatives = sum(1 for x in nums if x < 0)
    return -1 if negatives % 2 == 1 else 1

for nums, expected in examples:
    print(array_sign_count(nums), "| kỳ vọng", expected)

1 | kỳ vọng 1
0 | kỳ vọng 0
-1 | kỳ vọng -1


### Cách 4 — Nhân các dấu (theo gợi ý của đề)

Thay mỗi số bằng dấu của nó (`-1, 0, 1`) rồi nhân. Tích các dấu luôn nằm trong `{-1, 0, 1}` → không bao giờ tràn.

In [14]:
def array_sign_sign_product(nums: List[int]) -> int:
    result = 1
    for x in nums:
        result *= sign_func(x)
        if result == 0:
            break
    return result

print("dấu từng phần tử:", [sign_func(x) for x in [-1, -2, -3, -4, 3, 2, 1]])
for nums, expected in examples:
    print(array_sign_sign_product(nums), "| kỳ vọng", expected)

dấu từng phần tử: [-1, -1, -1, -1, 1, 1, 1]
1 | kỳ vọng 1
0 | kỳ vọng 0
-1 | kỳ vọng -1


### Cách 5 — `math.prod` một dòng (chỉ an toàn ở Python)

In [15]:
def array_sign_prod(nums: List[int]) -> int:
    p = math.prod(nums)
    return (p > 0) - (p < 0)

print("(True) - (False) =", True - False, "| (False) - (True) =", False - True)
for nums, expected in examples:
    print(array_sign_prod(nums), "| kỳ vọng", expected)

(True) - (False) = 1 | (False) - (True) = -1
1 | kỳ vọng 1
0 | kỳ vọng 0
-1 | kỳ vọng -1


`(p > 0) - (p < 0)` là mẹo viết `signFunc` một dòng: `bool` được coi như `1`/`0` khi làm toán.

### 7.1. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Tràn số ở C++/Java? |
|---|---|---|---|---|
| 1. Nhân trực tiếp | tích rồi lấy dấu | `O(n)` phép nhân (số lớn dần → chậm hơn trong Python) | `O(n)` chữ số của tích | **có** |
| **2. Lật dấu (chính)** | `sign = -sign`, dừng ở 0 | `O(n)` | `O(1)` | không |
| 3. Đếm số âm | chẵn/lẻ | `O(n)` (2 lượt) | `O(1)` | không |
| 4. Nhân các dấu | tích ∈ {-1,0,1} | `O(n)` | `O(1)` | không |
| 5. `math.prod` | một dòng | như cách 1 | như cách 1 | có (nếu dịch sang) |

## 8. Code nộp LeetCode

In [16]:
class Solution:
    def arraySign(self, nums: List[int]) -> int:
        sign = 1
        for x in nums:
            if x == 0:
                return 0
            if x < 0:
                sign = -sign
        return sign

sol = Solution()
for nums, expected in examples:
    print(sol.arraySign(nums), "| kỳ vọng", expected)

1 | kỳ vọng 1
0 | kỳ vọng 0
-1 | kỳ vọng -1


## 9. Vì sao thuật toán đúng?

**Invariant:** trước khi xét phần tử thứ `i`, nếu chưa trả về thì `nums[0..i-1]` không có số 0 và `sign == signFunc(nums[0] × ... × nums[i-1])` (tích rỗng = 1).

- Khởi tạo: tích rỗng = 1, `sign = 1`. ✓
- Gặp `x == 0`: tích toàn mảng = 0 bất kể phần còn lại → trả 0 đúng.
- Gặp `x > 0`: nhân với số dương không đổi dấu → giữ `sign`. ✓
- Gặp `x < 0`: nhân với số âm đổi dấu → `sign = -sign`. ✓

Kết thúc vòng lặp: `sign = signFunc(tích toàn mảng)`.

## 10. Độ phức tạp

`n = len(nums)`.

- **Time:** `O(n)` — mỗi phần tử một phép so sánh; dừng sớm khi gặp 0.
- **Space:** `O(1)`.

So với nhân trực tiếp: mỗi phép nhân số lớn tốn thời gian tỉ lệ với số chữ số, nên tổng thời gian cách 1 thực ra lớn hơn tuyến tính (khoảng `O(n²)` theo số chữ số trong trường hợp xấu).

## 11. Bộ test

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.arraySign([-1, -2, -3, -4, 3, 2, 1]), 1)
check("đề 2", sol.arraySign([1, 5, 0, 2, -3]), 0)
check("đề 3", sol.arraySign([-1, 1, -1, 1, -1]), -1)
check("1 phần tử dương", sol.arraySign([7]), 1)
check("1 phần tử âm", sol.arraySign([-7]), -1)
check("1 phần tử 0", sol.arraySign([0]), 0)
check("0 ở cuối sau nhiều số âm", sol.arraySign([-1, -1, -1, 0]), 0)
check("rất nhiều số âm chẵn", sol.arraySign([-100] * 1000), 1)
check("rất nhiều số âm lẻ", sol.arraySign([-100] * 999), -1)

PASS đề 1: 1
PASS đề 2: 0
PASS đề 3: -1
PASS 1 phần tử dương: 1
PASS 1 phần tử âm: -1
PASS 1 phần tử 0: 0
PASS 0 ở cuối sau nhiều số âm: 0
PASS rất nhiều số âm chẵn: 1
PASS rất nhiều số âm lẻ: -1


In [18]:
import random
random.seed(1822)
solvers = [array_sign_product, array_sign, array_sign_count,
           array_sign_sign_product, array_sign_prod, Solution().arraySign]
for _ in range(3000):
    n = random.randint(1, 15)
    nums = [random.randint(-3, 3) for _ in range(n)]     # miền nhỏ -> hay có 0 và số âm
    answers = [f(nums) for f in solvers]
    assert len(set(answers)) == 1, (nums, answers)
print("6 cách khớp nhau trên 3000 mảng ngẫu nhiên (seed 1822)")

6 cách khớp nhau trên 3000 mảng ngẫu nhiên (seed 1822)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Khởi tạo tích bằng 0

`product = 0` → mọi phép nhân sau đều ra 0.

In [19]:
def wrong_init_zero(nums):
    product = 0
    for x in nums:
        product *= x
    return sign_func(product)

print("sai:", wrong_init_zero([-1, -2, -3, -4, 3, 2, 1]), "| đúng:", array_sign([-1, -2, -3, -4, 3, 2, 1]))

sai: 0 | đúng: 1


### Lỗi 2: Đếm số âm nhưng quên kiểm tra số 0

In [20]:
def wrong_forget_zero(nums):
    negatives = sum(1 for x in nums if x < 0)
    return -1 if negatives % 2 else 1

print("sai:", wrong_forget_zero([1, 5, 0, 2, -3]), "| đúng:", array_sign([1, 5, 0, 2, -3]))

sai: -1 | đúng: 0


### Lỗi 3: Lật dấu bằng `sign = -1` (gán cứng)

Gán `sign = -1` thay vì `sign = -sign` → hai số âm vẫn cho -1.

In [21]:
def wrong_assign(nums):
    sign = 1
    for x in nums:
        if x == 0:
            return 0
        if x < 0:
            sign = -1
    return sign

print("sai:", wrong_assign([-1, -2]), "| đúng:", array_sign([-1, -2]))

sai: -1 | đúng: 1


### Lỗi 4: Trả về chính `product` thay vì dấu của nó

Đề yêu cầu `signFunc(product)` (`-1`, `0`, `1`), không phải `144`.

### Lỗi 5: Chia để "chuẩn hóa" `x // abs(x)` khi `x == 0`

`0 // abs(0)` → `ZeroDivisionError`. Phải xử lý `0` trước.

In [22]:
try:
    x = 0
    print(x // abs(x))
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

ZeroDivisionError: integer division or modulo by zero


## 13. Mẹo nhận dạng pattern

- Đề hỏi **dấu / chẵn lẻ / có bằng 0 không** của một phép tính lớn → **đừng tính giá trị thật**, chỉ theo dõi thuộc tính cần thiết.
- "Tích của mảng" → cẩn thận số 0 và tràn số; nghĩ tới đếm số âm.
- "Lật trạng thái mỗi khi gặp điều kiện" → `flag = not flag` hoặc `sign = -sign`.
- Bài liên quan: 152 Maximum Product Subarray (theo dõi cả max và min vì số âm lật dấu), 1567 Maximum Length of Subarray With Positive Product, 238 Product of Array Except Self.

## 14. Biến thể

### 14.1. Dấu của tích từng **tiền tố** (prefix)

Trả về list `res` với `res[i] = signFunc(nums[0] × ... × nums[i])`. Lưu ý: khi đã gặp 0 thì mọi tiền tố sau đều 0.

In [23]:
def prefix_signs(nums: List[int]) -> List[int]:
    res, sign = [], 1
    for x in nums:
        if x == 0:
            sign = 0
        elif x < 0:
            sign = -sign
        res.append(sign)
    return res

print(prefix_signs([-1, 2, -3, 0, 5]))   # [-1, -1, 1, 0, 0]

[-1, -1, 1, 0, 0]


Mẹo: khi `sign = 0`, `-sign` vẫn là `0` → các tiền tố sau tự động giữ 0.

### 14.2. Subarray dài nhất có tích dương (LeetCode 1567 — ý tưởng)

Chia mảng theo các số 0; trong mỗi khối, nếu số lượng số âm chẵn thì cả khối dương; nếu lẻ thì bỏ phần từ đầu khối tới số âm đầu tiên, hoặc từ số âm cuối cùng tới cuối khối.

In [24]:
def longest_positive_product(nums: List[int]) -> int:
    best = 0
    start = 0
    nums = nums + [0]                     # gác để chốt khối cuối
    for i, x in enumerate(nums):
        if x == 0:
            block = nums[start:i]
            neg_pos = [j for j, v in enumerate(block) if v < 0]
            if len(neg_pos) % 2 == 0:
                best = max(best, len(block))
            else:
                best = max(best, len(block) - neg_pos[0] - 1, neg_pos[-1])
            start = i + 1
    return best

print(longest_positive_product([1, -2, -3, 4]), longest_positive_product([0, 1, -2, -3, -4]),
      longest_positive_product([-1, -2, -3, 0, 1]))   # 4 3 2

4 3 2


## 15. Bài tập luyện tập

**Bài 1 (tính tay):** Không tính tích, cho biết `arraySign([3, -7, 2, -1, -9, 4])` và `arraySign([-2, 0, -2])`.

**Bài 2 (đoán output):** `(5 > 0) - (5 < 0)`, `(-3 > 0) - (-3 < 0)`, `(0 > 0) - (0 < 0)` là gì?

**Bài 3 (điền TODO):** Hoàn thiện lời giải lật dấu.

In [25]:
def ex3_array_sign(nums: List[int]) -> int:
    # TODO 1: sign = 1
    # TODO 2: gặp 0 -> return 0 ; gặp số âm -> sign = -sign
    # TODO 3: return sign
    pass

# assert ex3_array_sign([-1, -2, -3, -4, 3, 2, 1]) == 1
# assert ex3_array_sign([1, 5, 0, 2, -3]) == 0

**Bài 4 (tự viết):** Viết `ex4_sign_of_sum_of_products(a, b)` — dấu của `prod(a) + prod(b)`? (Gợi ý: lần này cần cả độ lớn; Python có số nguyên lớn nên được phép tính trực tiếp, nhưng hãy dùng `arraySign` để **thoát sớm** khi hai tích cùng dấu.)

In [26]:
def ex4_sign_of_sum_of_products(a: List[int], b: List[int]) -> int:
    # TODO
    pass

# assert ex4_sign_of_sum_of_products([2, 3], [-1, 6]) == 0
# assert ex4_sign_of_sum_of_products([-2], [-5]) == -1
# assert ex4_sign_of_sum_of_products([2, 2], [-3]) == 1

**Bài 5 (biến thể):** Viết `ex5_sign_after_removal(nums)` — trả về list: phần tử thứ `i` là dấu của tích mảng **sau khi bỏ** `nums[i]`. Yêu cầu `O(n)`: đếm trước số 0 và số âm.

In [27]:
def ex5_sign_after_removal(nums: List[int]) -> List[int]:
    # TODO
    pass

# assert ex5_sign_after_removal([-1, 2, -3]) == [-1, 1, -1]
# assert ex5_sign_after_removal([0, -2, 3]) == [-1, 0, 0]
# assert ex5_sign_after_removal([0, 0, 1]) == [0, 0, 0]

**Bài 6 (mini challenge):** Viết `ex6_count_positive_subarrays(nums)` — đếm số subarray (liên tiếp) có tích **dương**, trong `O(n)`. Gợi ý: duyệt từng khối không chứa 0; giữ số tiền tố có dấu `+` và dấu `-` đã gặp.

In [28]:
def ex6_count_positive_subarrays(nums: List[int]) -> int:
    # TODO
    pass

# assert ex6_count_positive_subarrays([1, -2, 3]) == 2
# assert ex6_count_positive_subarrays([-1, -1]) == 1
# assert ex6_count_positive_subarrays([2, 0, 3]) == 2

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm số âm, xem có 0 | 3 số âm (lẻ) | `-1` và `0` |
| 2 | `True` = 1, `False` = 0 | `1 - 0`, `0 - 1`, `0 - 0` | `1, -1, 0` |
| 3 | mục 5 | `if x == 0: return 0` | `sign = -sign` |
| 4 | cùng dấu → dấu chung | khác dấu → tính thật | `sign_func(math.prod(a) + math.prod(b))` |
| 5 | đếm `zeros`, `negs` | `zeros >= 2` → toàn 0 | `zeros == 1`: chỉ vị trí số 0 khác 0 |
| 6 | dấu tiền tố | subarray `(i, j]` dương ⇔ hai tiền tố cùng dấu | `count += pos` hoặc `neg` tùy dấu hiện tại |

## 17. Đáp án tham khảo

**Bài 1:** `[3, -7, 2, -1, -9, 4]` có 3 số âm, không có 0 → `-1`. `[-2, 0, -2]` → `0`.

**Bài 2:** `1`, `-1`, `0`.

In [29]:
print(array_sign([3, -7, 2, -1, -9, 4]), array_sign([-2, 0, -2]))
print((5 > 0) - (5 < 0), (-3 > 0) - (-3 < 0), (0 > 0) - (0 < 0))

def ex3_array_sign_ans(nums):
    sign = 1
    for x in nums:
        if x == 0:
            return 0
        if x < 0:
            sign = -sign
    return sign

def ex4_sign_of_sum_of_products_ans(a, b):
    sa, sb = ex3_array_sign_ans(a), ex3_array_sign_ans(b)
    if sa == sb or sb == 0:
        return sa
    if sa == 0:
        return sb
    return sign_func(math.prod(a) + math.prod(b))

def ex5_sign_after_removal_ans(nums):
    zeros = sum(1 for x in nums if x == 0)
    negs = sum(1 for x in nums if x < 0)
    res = []
    for x in nums:
        z = zeros - (x == 0)
        ng = negs - (x < 0)
        res.append(0 if z > 0 else (-1 if ng % 2 else 1))
    return res

def ex6_count_positive_subarrays_ans(nums):
    total = 0
    pos, neg = 1, 0      # tiền tố rỗng có dấu +
    sign = 1
    for x in nums:
        if x == 0:
            pos, neg, sign = 1, 0, 1    # bắt đầu khối mới
            continue
        if x < 0:
            sign = -sign
        if sign == 1:
            total += pos
            pos += 1
        else:
            total += neg
            neg += 1
    return total

import random
random.seed(18220)

def brute_count_positive(nums):
    return sum(1 for i in range(len(nums)) for j in range(i, len(nums)) if math.prod(nums[i:j + 1]) > 0)

assert ex3_array_sign_ans([-1, -2, -3, -4, 3, 2, 1]) == 1
assert ex3_array_sign_ans([1, 5, 0, 2, -3]) == 0
assert ex4_sign_of_sum_of_products_ans([2, 3], [-1, 6]) == 0
assert ex4_sign_of_sum_of_products_ans([-2], [-5]) == -1
assert ex4_sign_of_sum_of_products_ans([2, 2], [-3]) == 1
assert ex5_sign_after_removal_ans([-1, 2, -3]) == [-1, 1, -1]
assert ex5_sign_after_removal_ans([0, -2, 3]) == [-1, 0, 0]
assert ex5_sign_after_removal_ans([0, 0, 1]) == [0, 0, 0]
assert ex6_count_positive_subarrays_ans([1, -2, 3]) == 2
assert ex6_count_positive_subarrays_ans([-1, -1]) == 1
assert ex6_count_positive_subarrays_ans([2, 0, 3]) == 2
for _ in range(500):
    arr = [random.randint(-2, 2) for _ in range(random.randint(1, 8))]
    assert ex6_count_positive_subarrays_ans(arr) == brute_count_positive(arr), arr
print("Đáp án tham khảo PASS toàn bộ test (kể cả đối chiếu brute force cho bài 6)!")

-1 0
1 -1 0
Đáp án tham khảo PASS toàn bộ test (kể cả đối chiếu brute force cho bài 6)!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def arraySign(self, nums: List[int]) -> int:
        sign = 1
        for x in nums:
            if x == 0:
                return 0          # có 0 -> tích bằng 0
            if x < 0:
                sign = -sign      # mỗi số âm lật dấu một lần
        return sign
```

**Câu nhớ:** *"Có 0 thì 0; số âm chẵn thì 1; lẻ thì -1. Đừng nhân số thật."*

**Checklist:**

- [ ] Kiểm tra số 0 (và dừng sớm)?
- [ ] Lật dấu bằng `sign = -sign`, không gán cứng `-1`?
- [ ] Trả về `-1/0/1`, không phải giá trị tích?
- [ ] Không phụ thuộc vào việc tính tích khổng lồ (an toàn khi dịch sang C++/Java)?
- [ ] Tên method `arraySign`, trả `int`?